# 01 — Train text models

**Runtime → T4 GPU.** Copies only `src/configs/notebooks/requirements` (skips `.venv`).

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["HF_HUB_DISABLE_XET"] = "1"

os.chdir("/content")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE = Path("/content/drive/MyDrive/audio-sentiment-analysis")
assert DRIVE.exists(), DRIVE
print("Drive files:", sorted(x.name for x in DRIVE.iterdir()))

WORK = Path("/content/audio-sentiment-analysis")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

# Copy ONLY what training needs — never .venv / models / data blobs
KEEP_FILES = ["requirements.txt", "README.md", "TRAINING.md"]
KEEP_DIRS = ["src", "configs", "notebooks"]

for name in KEEP_FILES:
    src = DRIVE / name
    if src.exists():
        shutil.copy2(src, WORK / name)

for name in KEEP_DIRS:
    src = DRIVE / name
    if src.exists():
        shutil.copytree(
            src,
            WORK / name,
            ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".ipynb_checkpoints", ".venv"),
        )

# optional tiny text csvs only (skip audio wavs)
text_data = DRIVE / "data" / "text"
if text_data.exists():
    (WORK / "data" / "text").mkdir(parents=True, exist_ok=True)
    for f in text_data.glob("*.csv"):
        shutil.copy2(f, WORK / "data" / "text" / f.name)

os.chdir(WORK)
print("work files:", sorted(x.name for x in WORK.iterdir()))
assert (WORK / "requirements.txt").exists()
assert (WORK / "src" / "text").exists()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([
    sys.executable, "-c",
    "import nltk; nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('omw-1.4'); nltk.download('punkt'); nltk.download('punkt_tab')",
])

import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then re-run"

In [ ]:
import os
os.chdir("/content/audio-sentiment-analysis")
!python -m src.text.train_single_label --language en --mode transformer --epochs 3 --batch-size 16

In [ ]:
import os
os.chdir("/content/audio-sentiment-analysis")
!python -m src.text.train_multilabel --mode both --epochs 4 --batch-size 16

In [ ]:
import os, shutil
from pathlib import Path

os.chdir("/content/audio-sentiment-analysis")
DRIVE_FINAL = Path("/content/drive/MyDrive/audio-sentiment-analysis/models_final/text")
if DRIVE_FINAL.exists():
    shutil.rmtree(DRIVE_FINAL)
DRIVE_FINAL.mkdir(parents=True, exist_ok=True)

copied = []
for pattern in [
    "models/text/**/best/**",
    "models/text/**/*.joblib",
    "models/text/**/*metrics.json",
    "models/text/**/label_names.json",
]:
    for p in Path('.').glob(pattern):
        if not p.is_file() or 'runs' in p.parts:
            continue
        dest = DRIVE_FINAL / p.relative_to('models/text')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
        copied.append(str(dest))

print(f"Copied {len(copied)} files -> {DRIVE_FINAL}")
!du -sh "{DRIVE_FINAL}"